# CodonFM SAE — Quickstart

This notebook loads a trained Sparse Autoencoder for CodonFM (a codon language model) and runs basic health checks. By the end you'll know whether your SAE is working: how many features are alive, what fraction of variance it explains, and what individual feature activations look like.

## Setup

Paths need to be configured for your environment. Update the checkpoint, model, and data paths below to match your local setup.

In [ ]:
# ── Configure these paths for your environment ──
SAE_CHECKPOINT = "../outputs/1b_layer16/checkpoints/checkpoint_final.pt"
MODEL_PATH = "../../../../../../../checkpoints/NV-CodonFM-Encodon-TE-Cdwt-1B-v1"
CSV_PATH = "../../../../../../../codonfm/data/codonfm_ref_only.csv"
LAYER = 16
CONTEXT_LENGTH = 2048
BATCH_SIZE = 8
NUM_SEQUENCES = 500  # Keep small for quick iteration
DEVICE = "cuda"

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch


# Add recipe paths
_REPO_ROOT = Path("..").resolve().parent.parent.parent.parent.parent
_CODONFM_TE_DIR = _REPO_ROOT / "recipes" / "codonfm_ptl_te"
sys.path.insert(0, str(_CODONFM_TE_DIR))
sys.path.insert(0, str(Path("..").resolve()))

from codonfm_sae.data import read_codon_csv
from codonfm_sae.eval import evaluate_codonfm_loss_recovered
from sae.architectures import TopKSAE
from sae.utils import set_seed
from src.data.preprocess.codon_sequence import process_item
from src.inference.encodon import EncodonInference


set_seed(42)
device = DEVICE if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## Load SAE Checkpoint

Load the trained SAE from a checkpoint file. The checkpoint contains the model weights, architecture config (`input_dim`, `hidden_dim`, `top_k`), and training metadata.

In [ ]:
ckpt = torch.load(SAE_CHECKPOINT, map_location="cpu", weights_only=False)
state_dict = ckpt["model_state_dict"]
if any(k.startswith("module.") for k in state_dict):
    state_dict = {k.removeprefix("module."): v for k, v in state_dict.items()}

input_dim = ckpt.get("input_dim") or state_dict["encoder.weight"].shape[1]
hidden_dim = ckpt.get("hidden_dim") or state_dict["encoder.weight"].shape[0]
model_config = ckpt.get("model_config", {})
top_k = model_config.get("top_k")

sae = TopKSAE(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    top_k=top_k,
    normalize_input=model_config.get("normalize_input", False),
)
sae.load_state_dict(state_dict)
sae = sae.eval().to(device)

print(f"SAE: {input_dim} \u2192 {hidden_dim:,} features (top-{top_k})")
print(f"Expansion factor: {hidden_dim / input_dim:.1f}x")

## Load Encodon Model & Data

Load the CodonFM Encodon model (the base language model the SAE was trained on) and a CSV of codon sequences.

In [ ]:
inference = EncodonInference(
    model_path=MODEL_PATH,
    task_type="embedding_prediction",
    use_transformer_engine=True,
)
inference.configure_model()
inference.model.to(device).eval()

num_layers = len(inference.model.model.layers)
target_layer = LAYER if LAYER >= 0 else num_layers + LAYER
print(f"Encodon: {num_layers} layers, target layer {target_layer}")

# Load sequences
records = read_codon_csv(CSV_PATH, max_sequences=NUM_SEQUENCES, max_codons=CONTEXT_LENGTH - 2)
sequences = [r.sequence for r in records]
print(f"Loaded {len(sequences)} sequences")

## Loss Recovered

The primary quality metric for an SAE. Measures what fraction of the language model's cross-entropy loss is preserved when hidden states are routed through the SAE. A value of 1.0 means perfect reconstruction; 0.0 means the SAE output is no better than zero. Good SAEs typically achieve 0.85–0.95.

In [ ]:
result = evaluate_codonfm_loss_recovered(
    sae=sae,
    inference=inference,
    sequences=sequences[:200],  # Use subset for speed
    layer=LAYER,
    context_length=CONTEXT_LENGTH,
    batch_size=BATCH_SIZE,
    device=device,
)

print(f"Loss recovered:  {result.loss_recovered:.4f}")
print(f"CE (original):   {result.ce_original:.4f}")
print(f"CE (with SAE):   {result.ce_sae:.4f}")
print(f"CE (zero ablation): {result.ce_zero:.4f}")
print(f"Tokens evaluated: {result.n_tokens:,}")

## Feature Activation Statistics

Extract activations and compute basic per-feature statistics. Key things to check: what fraction of features are "alive" (fire at least once), the distribution of activation frequencies, and the sparsity pattern.

In [ ]:
from tqdm import tqdm


# Stream through sequences, compute per-feature max and firing counts
n_features = sae.hidden_dim
fire_counts = np.zeros(n_features, dtype=np.int64)
max_activations = np.zeros(n_features, dtype=np.float32)
total_tokens = 0

with torch.no_grad():
    for i in tqdm(range(0, len(sequences), BATCH_SIZE), desc="Extracting"):
        batch_seqs = sequences[i : i + BATCH_SIZE]
        items = [process_item(s, context_length=CONTEXT_LENGTH, tokenizer=inference.tokenizer) for s in batch_seqs]
        batch = {
            "input_ids": torch.tensor(np.stack([it["input_ids"] for it in items])).to(device),
            "attention_mask": torch.tensor(np.stack([it["attention_mask"] for it in items])).to(device),
        }
        out = inference.model(batch, return_hidden_states=True)
        hidden = out.all_hidden_states[LAYER]

        for j, it in enumerate(items):
            seq_len = it["attention_mask"].sum()
            emb = hidden[j, 1 : seq_len - 1, :].float()  # Strip CLS/SEP
            codes = sae.encode(emb)  # [n_codons, n_features]
            active = (codes > 0).cpu().numpy()
            fire_counts += active.sum(axis=0)
            np.maximum(max_activations, codes.max(dim=0).values.cpu().numpy(), out=max_activations)
            total_tokens += emb.shape[0]

        del out, hidden, batch

alive = fire_counts > 0
n_alive = alive.sum()
n_dead = n_features - n_alive
freq = fire_counts / total_tokens

print(f"Total tokens: {total_tokens:,}")
print(f"Alive features: {n_alive:,} / {n_features:,} ({n_alive / n_features:.1%})")
print(f"Dead features:  {n_dead:,} ({n_dead / n_features:.1%})")

## Activation Frequency Distribution

How often each feature fires. A healthy SAE has a broad range of frequencies — some features fire rarely (specialized detectors) and some fire often (common patterns). A spike at zero means dead features.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log-scale histogram of activation frequencies (alive only)
alive_freq = freq[alive]
axes[0].hist(np.log10(alive_freq + 1e-10), bins=50, color="#76b900", edgecolor="white", alpha=0.8)
axes[0].set_xlabel("log\u2081\u2080(activation frequency)")
axes[0].set_ylabel("Number of features")
axes[0].set_title(f"Activation Frequency Distribution ({n_alive:,} alive features)")
axes[0].axvline(
    np.log10(np.median(alive_freq)), color="red", linestyle="--", label=f"Median: {np.median(alive_freq):.4f}"
)
axes[0].legend()

# Max activation distribution
alive_max = max_activations[alive]
axes[1].hist(alive_max, bins=50, color="#0074DF", edgecolor="white", alpha=0.8)
axes[1].set_xlabel("Max activation value")
axes[1].set_ylabel("Number of features")
axes[1].set_title("Max Activation Distribution")
axes[1].axvline(np.median(alive_max), color="red", linestyle="--", label=f"Median: {np.median(alive_max):.1f}")
axes[1].legend()

plt.tight_layout()
plt.show()

## Inspect Individual Features

Pick a few features and look at what codons they activate on. This gives intuition for what the SAE learned — whether features correspond to recognizable biological patterns.

In [ ]:
# Pick the top 5 most frequently firing features
top_feature_ids = np.argsort(freq)[-5:][::-1]

CODON_TO_AA = {
    "TTT": "F",
    "TTC": "F",
    "TTA": "L",
    "TTG": "L",
    "CTT": "L",
    "CTC": "L",
    "CTA": "L",
    "CTG": "L",
    "ATT": "I",
    "ATC": "I",
    "ATA": "I",
    "ATG": "M",
    "GTT": "V",
    "GTC": "V",
    "GTA": "V",
    "GTG": "V",
    "TCT": "S",
    "TCC": "S",
    "TCA": "S",
    "TCG": "S",
    "CCT": "P",
    "CCC": "P",
    "CCA": "P",
    "CCG": "P",
    "ACT": "T",
    "ACC": "T",
    "ACA": "T",
    "ACG": "T",
    "GCT": "A",
    "GCC": "A",
    "GCA": "A",
    "GCG": "A",
    "TAT": "Y",
    "TAC": "Y",
    "TAA": "*",
    "TAG": "*",
    "CAT": "H",
    "CAC": "H",
    "CAA": "Q",
    "CAG": "Q",
    "AAT": "N",
    "AAC": "N",
    "AAA": "K",
    "AAG": "K",
    "GAT": "D",
    "GAC": "D",
    "GAA": "E",
    "GAG": "E",
    "TGT": "C",
    "TGC": "C",
    "TGA": "*",
    "TGG": "W",
    "CGT": "R",
    "CGC": "R",
    "CGA": "R",
    "CGG": "R",
    "AGT": "S",
    "AGC": "S",
    "AGA": "R",
    "AGG": "R",
    "GGT": "G",
    "GGC": "G",
    "GGA": "G",
    "GGG": "G",
}

# Use decoder weights to find which codons each feature promotes
lm_head = inference.model.model.lm_head.weight.data.float()  # [vocab, D]
decoder = sae.decoder.weight.data.float().to(device)  # [input_dim, hidden_dim]
mean_acts = sae.pre_bias.data.float().to(device) if hasattr(sae, "pre_bias") else torch.zeros(input_dim, device=device)

tokenizer = inference.tokenizer
codon_tokens = {}
for codon in CODON_TO_AA:
    tok_id = tokenizer.token_to_id(codon)
    if tok_id is not None:
        codon_tokens[codon] = tok_id

for feat_id in top_feature_ids:
    feat_dir = decoder[:, feat_id]
    logits = lm_head @ feat_dir  # [vocab]
    # Baseline: logits from mean
    baseline = lm_head @ mean_acts
    logit_diff = logits - baseline

    codon_logits = {c: float(logit_diff[tid]) for c, tid in codon_tokens.items()}
    sorted_codons = sorted(codon_logits.items(), key=lambda x: x[1], reverse=True)

    top_pos = sorted_codons[:5]
    top_neg = sorted_codons[-5:][::-1]

    print(f"\n{'=' * 50}")
    print(f"Feature {feat_id}  |  freq={freq[feat_id]:.4f}  |  max_act={max_activations[feat_id]:.1f}")
    print(f"  Top promoted:  {', '.join(f'{c}({CODON_TO_AA[c]})={v:.2f}' for c, v in top_pos)}")
    print(f"  Top suppressed: {', '.join(f'{c}({CODON_TO_AA[c]})={v:.2f}' for c, v in top_neg)}")

## Next Steps

Now that you've verified the SAE is working, explore deeper analyses:

- **02_codon_analysis.ipynb** — Compute codon usage metrics (CAI, tAI, RSCU) per feature
- **03_gene_enrichment.ipynb** — Run GSEA to find which biological pathways each feature detects
- **04_dashboard.ipynb** — Export data and launch the interactive dashboard
- **05_auto_interp.ipynb** — Use an LLM to automatically describe each feature